In [39]:
from pyspark.sql import SparkSession
import pyspark
print(pyspark.__version__)
from pyspark.sql import functions as F
from urllib.parse import urlparse
from datetime import date
from pyspark.sql.types import LongType
from py4j.java_gateway import java_import
from hdfs import InsecureClient

3.4.1


In [33]:
spark = (SparkSession.builder
    .appName("preprocess_batch")
    .master("spark://spark-master:7077")
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .enableHiveSupport()
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.hadoop.hive.exec.dynamic.partition", "true")
    .config("spark.hadoop.hive.exec.dynamic.partition.mode", "nonstrict")
    .getOrCreate())

spark.sql("USE DATABASE CryptoPredictions")

25/12/16 19:24:59 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


DataFrame[]

In [34]:
BASE_PATH = "hdfs://namenode:8020/nifi/stock-prices/"
META_PATH = "hdfs://namenode:8020/nifi/metadata/stock_prices_last_path.txt"

paths_to_read = find_new_paths(spark, BASE_PATH, META_PATH)

['2025-11-27', '2025-11-28', '2025-11-29', '2025-11-30', '2025-12-01', '2025-12-02', '2025-12-03', '2025-12-04', '2025-12-05', '2025-12-06', '2025-12-07', '2025-12-08', '2025-12-09', '2025-12-10', '2025-12-11', '2025-12-12', '2025-12-13', '2025-12-14', '2025-12-15']


In [43]:
folder = "hdfs://namenode:8020/nifi/stock-prices"

# Wczytanie całego folderu do DataFrame
df = spark.read.parquet(*paths_to_read)

# Spark zapisuje plik źródłowy w kolumnie __file__ (od Spark 3.x)
df_with_file = df.withColumn("__file__", F.input_file_name())

# Pobranie schematów per plik
files = df_with_file.select("__file__").distinct().collect()

bad_files = []
for row in files:
    file_path = row["__file__"]
    df_file = spark.read.parquet(file_path)
    dtype = df_file.schema["lastVolume"].dataType
    if not isinstance(dtype, LongType):
        bad_files.append(file_path)
        print(f"{file_path} -> lastVolume: {dtype}")

# Access Hadoop FileSystem
conf = spark._jsc.hadoopConfiguration()

uri = spark._jvm.java.net.URI(BASE_PATH)
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(uri, conf)

bad_files_folder = "hdfs://namenode:8020/nifi/bad-stock-prices"
bad_folder_path = spark._jvm.Path(bad_files_folder)

if not fs.exists(bad_folder_path):
    fs.mkdirs(bad_folder_path)
    
for file_path in bad_files:
    src = spark._jvm.Path(file_path)
    # Construct destination path (keep original filename)
    filename = file_path.split("/")[-1]
    dst = spark._jvm.Path(f"{bad_files_folder}/{filename}")
    fs.rename(src, dst)  # moves the file

# The dataframe needs to be reloaded if there were any bad files
if bad_files:
    df = spark.read.parquet(*paths_to_read)

In [44]:
def transform_index_snapshot(df):
    res = (
        df
        # Dopasowanie nazw
        .withColumn("IndexName", F.col("exchange"))
        .withColumn("Datetime", F.to_timestamp("fetch_timestamp"))
        .withColumn("CurrentPrice", F.col("lastPrice"))
        .withColumn("CurrentVolume", F.col("lastVolume"))
        .withColumn("OpeningPrice", F.col("open"))
        .withColumn("LowestDayPrice", F.col("dayLow"))
        .withColumn("HighestDayPrice", F.col("dayHigh"))
        .withColumn("LowestYearlyPrice", F.col("yearLow"))
        .withColumn("HighestYearlyPrice", F.col("yearHigh"))
        .withColumn("FiftyDayAveragePrice", F.col("fiftyDayAverage"))
        .withColumn("TenDayAverageVolume", F.col("tenDayAverageVolume"))
        .withColumn("ThreeMonthAverageVolume", F.col("threeMonthAverageVolume"))
        .withColumn("TwoHundredDaysAveragePrice", F.col("twoHundredDayAverage"))
        .withColumn("YearOverYearPriceChange", F.col("yearChange"))
        # przeniosłem return na koniec -> tu przeszkadza we wczytywaniu nowych danych do hive.
        .withColumn("PartitionDate", F.to_date("fetch_timestamp"))
    )

    final_cols = [
        "IndexName",
        "Datetime",
        "CurrentPrice",
        "CurrentVolume",
        "OpeningPrice",
        "LowestDayPrice",
        "HighestDayPrice",
        "LowestYearlyPrice",
        "HighestYearlyPrice",
        "FiftyDayAveragePrice",
        "TwoHundredDaysAveragePrice",
        "TenDayAverageVolume",
        "ThreeMonthAverageVolume",
        "YearOverYearPriceChange",
        "PartitionDate"
    ]

    return res.select(*final_cols)

In [45]:
df_transformed = transform_index_snapshot(df)
print(df_transformed.count())
df_transformed.show(1, vertical=True, truncate=False)
df_transformed.agg(
    F.min("Datetime").alias("min_datetime"),
    F.max("Datetime").alias("max_datetime")
).show()

77760
-RECORD 0-----------------------------------------
 IndexName                  | SNP                 
 Datetime                   | 2025-12-05 16:00:15 
 CurrentPrice               | 6881.43017578125    
 CurrentVolume              | 834955947           
 OpeningPrice               | 6866.31982421875    
 LowestDayPrice             | 6866.31982421875    
 HighestDayPrice            | 6895.77978515625    
 LowestYearlyPrice          | 4835.0400390625     
 HighestYearlyPrice         | 6920.33984375       
 FiftyDayAveragePrice       | 6738.8933984375     
 TwoHundredDaysAveragePrice | 6191.572690429687   
 TenDayAverageVolume        | 4835350000          
 ThreeMonthAverageVolume    | 5459956093          
 YearOverYearPriceChange    | 0.12591397346866276 
 PartitionDate              | 2025-12-05          
only showing top 1 row



[Stage 878:===================================================>   (13 + 1) / 14]

+-------------------+-------------------+
|       min_datetime|       max_datetime|
+-------------------+-------------------+
|2025-11-27 15:00:07|2025-12-15 23:59:26|
+-------------------+-------------------+



In [46]:
(df_transformed.write
    .mode("append")
    .format("hive")
    .partitionBy("PartitionDate")
    .saveAsTable("IndexSnapshot"))

latest_dir = max(new_dirs)

spark.createDataFrame(
    [(latest_dir,)],
    ["last_processed_path"]
).write.mode("overwrite").text(META_PATH)

print(f"Created last path file: {latest_dir}")

25/12/16 19:54:47 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
                                                                                

Created last path file: 2025-12-15


In [30]:
spark.sql("""SELECT * FROM IndexSnapshot
            LIMIT 6""").show(vertical=True, truncate=False)

-RECORD 0------------------------------------------
 IndexName                  | SNP                  
 Datetime                   | 2025-11-19 15:00:47  
 CurrentPrice               | 6671.7099609375      
 CurrentVolume              | 362266710            
 OpeningPrice               | 6625.83984375        
 LowestDayPrice             | 6618.47998046875     
 HighestDayPrice            | 6675.14990234375     
 LowestYearlyPrice          | 4835.0400390625      
 HighestYearlyPrice         | 6920.33984375        
 FiftyDayAveragePrice       | 6709.807392578125    
 TwoHundredDaysAveragePrice | 6154.74173828125     
 TenDayAverageVolume        | 5444071000           
 ThreeMonthAverageVolume    | 5376607538           
 YearOverYearPriceChange    | 0.11833648134246547  
 PartitionDate              | 2025-11-19           
-RECORD 1------------------------------------------
 IndexName                  | DJI                  
 Datetime                   | 2025-11-19 15:00:49  
 CurrentPric

# Feature creation

### Notka od Łukasza:
Najlepiej zrób nowy notebook na tworzenie ficzerów.

In [21]:
from pyspark.sql import functions as F

df_transformed = df_transformed.withColumn("return", F.when(F.col("OpeningPrice") != 0,(F.col("CurrentPrice") - F.col("OpeningPrice")) / F.col("OpeningPrice")).otherwise(None))  
df_left = df_transformed.withColumn(
    "Datetime_1h_ago",
    F.col("Datetime") - F.expr("INTERVAL 1 HOUR")
)

df_right = df_transformed.select(
    F.col("IndexName").alias("idx"),
    F.col("Datetime").alias("dt"),
    F.col("CurrentPrice").alias("Price_1HourAgo")
)

df_copy = (
    df_left
    .join(
        df_right,
        (df_left.IndexName == df_right.idx) &
        (df_left.Datetime_1h_ago == df_right.dt),
        how="left"
    )
    .drop("idx", "dt", "Datetime_1h_ago")
)
df_copy.show(5)

In [47]:
spark.stop()